In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
train_dir = "dataset/Training"
test_dir = "dataset/Testing"

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [4]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [5]:
print(train_data.class_indices)

{'glioma_tumor': 0, 'meningioma_tumor': 1, 'no_tumor': 2, 'pituitary_tumor': 3}


In [6]:
images, labels = next(train_data)

print(images.shape)   # (32, 224, 224, 3)
print(labels.shape)   # (32, 4)

(32, 224, 224, 3)
(32, 4)


CNN

In [7]:
import tensorflow as tf

In [8]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Load pretrained model
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [9]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 112, 112, 32)      │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 112, 112, 32)      │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 112, 112, 32)      │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 32)      │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 112, 112, 32)      │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 112, 112, 32)      │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 112, 112, 96)      │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 112, 112, 96)      │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 112, 112, 96)      │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 113, 113, 96)      │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 56, 56, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [11]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=15
)

Epoch 1/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 156s 2s/step - accuracy: 0.7596 - loss: 0.5982 - val_accuracy: 0.5635 - val_loss: 1.4321
Epoch 2/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.8446 - loss: 0.3877 - val_accuracy: 0.5736 - val_loss: 1.3537
Epoch 3/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.8700 - loss: 0.3311 - val_accuracy: 0.6320 - val_loss: 1.4371
Epoch 4/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 146s 2s/step - accuracy: 0.8774 - loss: 0.3079 - val_accuracy: 0.6726 - val_loss: 1.2594
Epoch 5/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.8930 - loss: 0.2838 - val_accuracy: 0.7107 - val_loss: 1.2123
Epoch 6/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.8951 - loss: 0.2582 - val_accuracy: 0.6802 - val_loss: 1.4754
Epoch 7/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.9115 - loss: 0.2300 - val_accuracy: 0.6827 - val_loss: 1.6285
Epoch 8/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.9132 - loss: 0.2267 - val_accuracy: 0.6904 - v

In [12]:
# Unfreeze last 30 layers
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5
)

Epoch 1/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 0.6003 - loss: 1.6982 - val_accuracy: 0.7284 - val_loss: 1.7435
Epoch 2/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.8345 - loss: 0.5652 - val_accuracy: 0.7386 - val_loss: 1.8795
Epoch 3/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 193s 2s/step - accuracy: 0.8711 - loss: 0.3900 - val_accuracy: 0.7411 - val_loss: 1.9626
Epoch 4/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.8760 - loss: 0.3312 - val_accuracy: 0.7259 - val_loss: 2.0198
Epoch 5/5
90/90 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.8892 - loss: 0.2972 - val_accuracy: 0.7411 - val_loss: 2.0622


In [13]:
loss, accuracy = model.evaluate(test_data)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.7411 - loss: 2.0622
Test Loss: 2.0622291564941406
Test Accuracy: 0.7411167621612549


In [14]:
model.save("brain_tumor_model.h5")

In [18]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load model
model = load_model("brain_tumor_model.h5")

# Class names (must match your dataset)
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Path to test image
img_path = "dataset/Testing/glioma_tumor/image(1).jpg"   # change to your image path

# Load and preprocess image
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
prediction = model.predict(img_array)

predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction)

# Output (no plot)
print("Prediction:", predicted_class)
print("Confidence: {:.2f}%".format(confidence * 100))

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Prediction: meningioma
Confidence: 97.05%
